# Robust susceptibility validation by Monte Carlo

`main_robust_suscept.ipynb` の robust-LCB 推奨点を validation する notebook です。GP posterior mean 上で finite-difference susceptibility を計算する代わりに、中心点 `x_center` の周りに Monte Carlo 摂動点を作り、HFSS 実測 `S11` から susceptibility を推定します。


In [48]:
from pathlib import Path
import json
import os
import time

import numpy as np
import pandas as pd

import lib_config as config
import lib_backbone
import lib_gp

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [49]:
# Load parameter metadata from _config.toml.
_config = config._loadConfig(Path("./_config.toml"))
app_config = config.initParams(_config, debug=True)

backbone = lib_backbone.Backbone(config=app_config)
base_dir = app_config.env.dir_base
cfg = app_config.hfss
ROUND_DECIMALS = app_config.runtime.round_decimals

LOWER_BOUNDS = np.asarray(cfg.lower_bounds, dtype=float)
UPPER_BOUNDS = np.asarray(cfg.upper_bounds, dtype=float)
BOUNDS = np.vstack([LOWER_BOUNDS, UPPER_BOUNDS]).T
PARAM_NAMES = list(cfg.param_names)

print("n_params:", cfg.n_params)
print("PARAM_NAMES:", PARAM_NAMES)
print("ROUND_DECIMALS:", ROUND_DECIMALS)



[Io]
  filename_input      : params.csv
  filename_output     : results.csv
  filename_temp       : temp_hfss_export.csv

[Opt]
  kernel_type         : Matern52
  length_scale        : 3.605550
  noise_std           : 0.010000
  noise_var           : 0.000100

[Hfss]
  n_simulation        : 100
  n_repeats           : 6
  n_init              : 20
  n_params            : 13
  lower_bounds        : [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 2, 2, 1]
  upper_bounds        : [10.0, 10.0, 10.0, 10.0, 10.0, 2.0, 2.0, 2.0, 2.0, 2.0, 7, 7, 6]
  param_names         : ['h1', 'h2', 'h3', 'h4', 'h5', 's1', 's2', 's3', 's4', 's5', 'a', 'b', 'k']
  param_units         : ['mm', 'mm', 'mm', 'mm', 'mm', '', '', '', '', '', 'mm', 'mm', '']
  filename_models     : ['Backshort.step', 'Finshape.step']
  param_groups        : {'A': {'param_names': ['h1', 'h2', 'h3', 'h4', 'h5', 's1', 's2', 's3', 's4', 's5'], 'param_units': ['mm', 'mm', 'mm', 'mm', 'mm', '', '', '', '', ''], 'lower_bounds': [1.0, 1.

In [50]:
# ===== User input =====
# cfg.n_params (= 13) と同じ長さの中心パラメータをここに入力してください。
# robust-penalty
x_center = np.asarray(
    [
        6.089399,
        6.290120,
        3.006193,
        7.122293,
        6.748895,
        1.024461,
        1.129244,
        1.777818,
        0.779463,
        0.225789,
        2.372480,
        3.652333,
        5.799660,
    ],
    dtype=float,
)

N_VALIDATION = 100
RUN_ALL_VALIDATION_ROWS = True  # False: HFSS debug 用に先頭 DEBUG_N_ROWS だけ評価
DEBUG_N_ROWS = 5
# PERTURBATION_STD_RATIO = 0.01  # legacy ratio-based std; kept for reference.
RANDOM_SEED = 101

# Custom per-parameter perturbation standard deviations.
# PARAM_NAMES is loaded from _config.toml in the previous cell, so this dictionary
# is converted to a diagonal covariance matrix in the same parameter order.
CUSTOM_PERTURBATION_STD = {
    "h1": 0.1,
    "h2": 0.1,
    "h3": 0.1,
    "h4": 0.1,
    "h5": 0.1,
    "s1": 0.01,
    "s2": 0.02,
    "s3": 0.02,
    "s4": 0.02,
    "s5": 0.02,
    "a": 0.1,
    "b": 0.1,
    "k": 0.1,
}

# Perturbation enable flags for each x_center component.
# Set False to keep that parameter fixed at x_center for all perturbation rows.
# Example: h1 を固定して他の変数だけ perturbation する場合は "h1": False にします。
PERTURBATION_ENABLED = {
    "h1": True,
    "h2": True,
    "h3": True,
    "h4": True,
    "h5": True,
    "s1": True,
    "s2": True,
    "s3": True,
    "s4": True,
    "s5": True,
    "a": True,
    "b": True,
    "k": True,
}

CUSTOM_PERTURBATION_STD_ARRAY = np.asarray(
    [
        CUSTOM_PERTURBATION_STD[name] if PERTURBATION_ENABLED[name] else 0.0
        for name in PARAM_NAMES
    ],
    dtype=float,
)
PERTURBATION_SIGMA = np.diag(CUSTOM_PERTURBATION_STD_ARRAY**2)
FIXED_PARAM_NAMES = [name for name in PARAM_NAMES if not PERTURBATION_ENABLED[name]]

In [51]:
def validate_x_center(x, n_params, lower_bounds, upper_bounds):
    """Validate and round the user-provided center vector."""
    x = np.asarray(x, dtype=float).reshape(-1)
    if x.size != n_params:
        raise ValueError(
            f"x_center must have length cfg.n_params={n_params}, but got length {x.size}. "
            "Set x_center before generating parametric_input.csv."
        )

    below = x < lower_bounds
    above = x > upper_bounds
    if np.any(below) or np.any(above):
        bad = [
            (PARAM_NAMES[i], float(x[i]), float(lower_bounds[i]), float(upper_bounds[i]))
            for i in np.where(below | above)[0]
        ]
        raise ValueError(f"x_center has out-of-bounds values: {bad}")

    return np.round(x, ROUND_DECIMALS)


def build_validation_sigma(custom_sigma=None):
    """Build validation covariance.

    By default this notebook uses ``PERTURBATION_SIGMA`` from the user-input
    cell. If ``custom_sigma`` is None, the legacy ratio-based formula can still
    be used by defining ``PERTURBATION_STD_RATIO`` in the user-input cell.
    """
    if custom_sigma is not None:
        sigma = np.asarray(custom_sigma, dtype=float)
    else:
        bounds_width = UPPER_BOUNDS - LOWER_BOUNDS
        perturbation_std = PERTURBATION_STD_RATIO * bounds_width
        sigma = np.diag(perturbation_std ** 2)

    expected_shape = (cfg.n_params, cfg.n_params)
    if sigma.shape != expected_shape:
        raise ValueError(f"Sigma must have shape {expected_shape}, got {sigma.shape}.")
    return sigma


In [52]:
x_center = validate_x_center(x_center, cfg.n_params, LOWER_BOUNDS, UPPER_BOUNDS)
VALIDATION_SIGMA = build_validation_sigma(PERTURBATION_SIGMA)
rng = np.random.default_rng(RANDOM_SEED)

# Generate only perturbation rows here. The center row is prepended as index 0 below.
Z_perturbation = lib_gp.sample_input_perturbations(
    x=x_center,
    Sigma=VALIDATION_SIGMA,
    n_samples=N_VALIDATION,
    bounds=BOUNDS,
    rng=rng,
)
Z_perturbation = np.round(Z_perturbation, ROUND_DECIMALS)

# Enforce fixed-parameter flags exactly after sampling/clipping.
if FIXED_PARAM_NAMES:
    fixed_indices = [PARAM_NAMES.index(name) for name in FIXED_PARAM_NAMES]
    Z_perturbation[:, fixed_indices] = x_center[fixed_indices]


# index 0: center, index 1..N_VALIDATION: perturbations
Z_validation_with_center = np.vstack([x_center, Z_perturbation])
parametric_input_df = pd.DataFrame(Z_validation_with_center, columns=PARAM_NAMES)
parametric_input_df.index.name = "sample_idx"

print("Fixed parameters:", FIXED_PARAM_NAMES if FIXED_PARAM_NAMES else "None")
print("DataFrame shape:", parametric_input_df.shape)
print("Center row index:", parametric_input_df.index[0])
print("Perturbation index range:", f"{parametric_input_df.index[1]}..{parametric_input_df.index[-1]}")
display(parametric_input_df.head())


Fixed parameters: None
DataFrame shape: (101, 13)
Center row index: 0
Perturbation index range: 1..100


,h1,h2,h3,h4,h5,s1,s2,s3,s4,s5,a,b,k
sample_idx,,,,,,,,,,,,,
0,6.089399,6.290120,3.006193,7.122293,6.748895,1.024461,1.129244,1.777818,0.779463,0.225789,2.372480,3.652333,5.799660
1,6.195479,6.364549,2.975224,7.159025,6.919934,1.007976,1.111973,1.797098,0.793616,0.239544,2.169017,3.712663,5.720645
2,6.080294,6.279086,3.170539,7.088280,6.628135,1.029471,1.129191,1.749634,0.796541,0.229653,2.328751,3.479481,5.766451
3,6.118891,6.358978,2.841951,6.978754,6.639097,1.031966,1.134121,1.785113,0.763718,0.251980,2.506607,3.801238,5.848233
4,5.938736,6.436419,3.158506,7.047545,6.564145,1.045885,1.147250,1.806986,0.758394,0.243432,2.231423,3.897526,5.833685


In [53]:
# Create a timestamped run directory using the same Backbone directory convention as the other notebooks.
backbone.mkdir()
run_dir = backbone._get_dir_run()

parametric_input_csv = run_dir / "parametric_input.csv"
parametric_input_df.to_csv(parametric_input_csv, index=False)

print(f"Saved parametric input CSV: {parametric_input_csv}")
print(f"Rows x columns: {parametric_input_df.shape[0]} x {parametric_input_df.shape[1]}")


Created new run directory: T:\RAkizawa\HFSS_C2WR10\src\0618191119
Saved parametric input CSV: T:\RAkizawa\HFSS_C2WR10\src\0618191119\parametric_input.csv
Rows x columns: 101 x 13


In [54]:
# Optional: inspect the saved CSV.
saved_parametric_input_df = pd.read_csv(parametric_input_csv)
display(saved_parametric_input_df.head())


,h1,h2,h3,h4,h5,s1,s2,s3,s4,s5,a,b,k
0,6.089399,6.290120,3.006193,7.122293,6.748895,1.024461,1.129244,1.777818,0.779463,0.225789,2.372480,3.652333,5.799660
1,6.195479,6.364549,2.975224,7.159025,6.919934,1.007976,1.111973,1.797098,0.793616,0.239544,2.169017,3.712663,5.720645
2,6.080294,6.279086,3.170539,7.088280,6.628135,1.029471,1.129191,1.749634,0.796541,0.229653,2.328751,3.479481,5.766451
3,6.118891,6.358978,2.841951,6.978754,6.639097,1.031966,1.134121,1.785113,0.763718,0.251980,2.506607,3.801238,5.848233
4,5.938736,6.436419,3.158506,7.047545,6.564145,1.045885,1.147250,1.806986,0.758394,0.243432,2.231423,3.897526,5.833685


## Run Monte Carlo validation simulations

GP optimization は行わず、生成済みの `parametric_input_df` を順番に評価します。`RUN_ALL_VALIDATION_ROWS=True` の場合は中心点を含む全行を実行し、`False` の場合は接続確認用に先頭 `DEBUG_N_ROWS` 行だけを実行します。


In [55]:
# Initialize the HDF5 storer in the same timestamped run directory.
# If the previous save cell already created run_dir, initStorer reuses it.
backbone.initStorer(mode="w")
run_dir = backbone._get_dir_run()

model_paths, model_paths_str = backbone._get_path_models()
RESULTS_FILE = str(run_dir / Path(_config["io"]["filename_output"]))
TEMP_FILE = str(run_dir / Path(_config["io"]["filename_temp"]))

print("Run directory:", run_dir)
print("HDF5 file:", run_dir / "results.h5")
print("RESULTS_FILE:", RESULTS_FILE)
print("TEMP_FILE:", TEMP_FILE)


HDF5 dataset created at: T:\RAkizawa\HFSS_C2WR10\src\0618191119\results.h5
Run directory: T:\RAkizawa\HFSS_C2WR10\src\0618191119
HDF5 file: T:\RAkizawa\HFSS_C2WR10\src\0618191119\results.h5
RESULTS_FILE: T:\RAkizawa\HFSS_C2WR10\src\0618191119\results.csv
TEMP_FILE: T:\RAkizawa\HFSS_C2WR10\src\0618191119\temp_hfss_export.csv


In [56]:
def round_params(params, decimals=ROUND_DECIMALS):
    return np.round(np.asarray(params, dtype=float).flatten(), decimals)


def round_history_row(row, param_names, decimals=ROUND_DECIMALS):
    rounded = dict(row)
    for name in param_names:
        if name in rounded:
            rounded[name] = float(np.round(rounded[name], decimals))
    for name in ("S11", "S11_min_freq_GHz", "Metric", "gamma"):
        if name in rounded and pd.notna(rounded[name]):
            rounded[name] = float(np.round(rounded[name], decimals))
    return rounded


S11_FREQUENCY_COLUMN = "Freq [GHz]"
S11_CURVE_PREFIX = "S11_sim"
s11_frequency_df = None


def _detect_frequency_s11_columns(df_temp):
    """Detect frequency and S11 columns from the HFSS temp CSV."""
    columns = list(df_temp.columns)
    freq_candidates = [column for column in columns if "freq" in str(column).lower()]
    freq_col = freq_candidates[0] if freq_candidates else columns[0]

    s11_candidates = [
        column
        for column in columns
        if column != freq_col and ("s(" in str(column).lower() or "s11" in str(column).lower())
    ]
    s11_col = s11_candidates[0] if s11_candidates else [column for column in columns if column != freq_col][0]
    return freq_col, s11_col


def getResult(input_params, param_names, temp_hfss_path, result_file_path, sim_id):
    """Read one HFSS frequency sweep and append scalar/curve outputs.

    The HFSS temp CSV is expected to contain a frequency column and an S11 column,
    e.g. ``Freq [GHz]`` and ``dB(S(Port1,Port1)) []``. The frequency grid is
    stored only once in ``s11_frequency_df``; each simulation appends a new S11
    curve column. For compatibility with the existing ``repeat_1`` summary format,
    this function also records the minimum S11 value and its frequency in
    ``results.csv``.
    """
    global s11_frequency_df

    df_temp = pd.read_csv(temp_hfss_path)
    header_flag = not os.path.exists(result_file_path)

    try:
        freq_col, s11_col = _detect_frequency_s11_columns(df_temp)
        freq_values = df_temp[freq_col].to_numpy(dtype=float)
        s11_values = df_temp[s11_col].to_numpy(dtype=float)

        if s11_frequency_df is None:
            s11_frequency_df = pd.DataFrame({S11_FREQUENCY_COLUMN: freq_values})
        else:
            existing_freq = s11_frequency_df[S11_FREQUENCY_COLUMN].to_numpy(dtype=float)
            if len(existing_freq) != len(freq_values) or not np.allclose(existing_freq, freq_values):
                raise ValueError("HFSS frequency grid changed between simulations.")

        curve_column = f"{S11_CURVE_PREFIX}_{int(sim_id):03d}"
        s11_frequency_df[curve_column] = s11_values

        min_idx = int(np.nanargmin(s11_values))
        result_row = dict(zip(param_names, input_params))
        result_row["S11"] = float(s11_values[min_idx])
        result_row["S11_min_freq_GHz"] = float(freq_values[min_idx])

        df_result = pd.DataFrame([result_row])
        df_result.to_csv(result_file_path, mode="a", header=header_flag, index=False)

        try:
            os.remove(temp_hfss_path)
        except OSError:
            pass
        return result_row

    except Exception as e:
        print(f"[Error][getResult] Failed to process result: {e}")
        raise


def target_hfss(_config_temp, sim_id, param_names, params):
    """Evaluate one parameter vector through the existing HFSS watcher workflow."""
    params = round_params(params)
    backbone.call_subroutine(
        _config_temp,
        sim_id,
        param_names,
        params,
        value_fmt=f"{{:.{ROUND_DECIMALS}f}}",
    )
    return getResult(params, param_names, TEMP_FILE, RESULTS_FILE, sim_id)


def costFunctionWrapper(param_names, params):
    params = round_params(params, decimals=ROUND_DECIMALS)
    sim_id = backbone._getSimulationID()
    result_row = target_hfss(_config_temp, sim_id, param_names, params)
    y = float(result_row["S11"])

    row = dict(result_row)
    row["Metric"] = np.nan
    row["gamma"] = np.nan
    row["routine_idx"] = 0
    return y, round_history_row(row, param_names)


In [57]:
# HFSS watcher run-specific config, matching main.ipynb's hand-off format.
rows_to_run = parametric_input_df if RUN_ALL_VALIDATION_ROWS else parametric_input_df.iloc[:DEBUG_N_ROWS]

_config_temp = {
    "n_simulation": int(len(rows_to_run)),
    "n_repeats": 1,
    "WATCH_DIR": str(run_dir),
    "INPUT_FILE": str(run_dir / Path(_config["io"]["filename_input"])),
    "MODEL_FILE": model_paths_str,
    "RESULTS_FILE": RESULTS_FILE,
    "TEMP_FILE": TEMP_FILE,
    "DONE_FLAG_FILE": str(run_dir / Path("hfss.done")),
}

done_flag_path = Path(_config_temp["DONE_FLAG_FILE"])
done_flag_path.unlink(missing_ok=True)

with open(base_dir / Path("_config_HFSS.json"), "w") as f:
    json.dump(_config_temp, f, indent=4)

print(f"Temporarily updated '{base_dir / Path('_config_HFSS.json')}' with run-specific WATCH_DIR for HFSS.")
print("Number of Monte Carlo validation simulations:", len(rows_to_run))


Temporarily updated 'T:\RAkizawa\HFSS_C2WR10\src\_config_HFSS.json' with run-specific WATCH_DIR for HFSS.
Number of Monte Carlo validation simulations: 101


In [ ]:
# Monte Carlo validation run: evaluate generated rows without GP optimization.
parametric_history = []
start = time.perf_counter()

try:
    for sample_idx, (_, row) in enumerate(rows_to_run.iterrows(), start=1):
        params = row[PARAM_NAMES].to_numpy(dtype=float)
        print(f"[MC validation] HFSS evaluation {sample_idx}/{len(rows_to_run)} (source rows: {len(parametric_input_df)})")
        _, result_row = costFunctionWrapper(PARAM_NAMES, params)
        result_row["sample_idx"] = sample_idx - 1
        result_row["is_center"] = (sample_idx == 1)
        parametric_history.append(result_row)

    df_final = pd.DataFrame(parametric_history)
    df_final[PARAM_NAMES] = df_final[PARAM_NAMES].round(ROUND_DECIMALS)
    df_final["S11"] = df_final["S11"].round(ROUND_DECIMALS)

    df_output = backbone._genOutputDataFrame(df_final)
    if "best" in df_output:
        df_output["best"] = df_output["best"].round(ROUND_DECIMALS)

    parametric_results_csv = run_dir / "parametric_results.csv"
    df_output.to_csv(parametric_results_csv, index=False)
    backbone._addNewDatasetToHDF(df_output.select_dtypes(include=[np.number]), "output", "repeat_1")

    if s11_frequency_df is not None:
        s11_frequency_csv = run_dir / "s11_frequency_response.csv"
        s11_frequency_df.to_csv(s11_frequency_csv, index=False)
        backbone._addNewDatasetToHDF(s11_frequency_df.select_dtypes(include=[np.number]), "output", "s11_frequency_response")

    elapsed = time.perf_counter() - start
    best_idx = df_output["S11"].idxmin()
    print("-" * 75)
    print(f"Monte Carlo validation finished in {elapsed:.3f} seconds")
    print(f"Best S11: {df_output.loc[best_idx, 'S11']:.10f}")
    print("Best row:")
    display(df_output.loc[[best_idx]])
    print(f"Saved results CSV: {parametric_results_csv}")
    if s11_frequency_df is not None:
        print(f"Saved S11 frequency response CSV: {s11_frequency_csv}")
    print(f"Saved HDF5: {run_dir / 'results.h5'}")

finally:
    done_flag_path = Path(_config_temp["DONE_FLAG_FILE"])
    done_flag_path.touch()

    json_file = base_dir / Path("_config_HFSS.json")
    json_file.unlink(missing_ok=True)
    if backbone.h5f:
        backbone.h5f.close()


[MC validation] HFSS evaluation 1/101 (source rows: 101)


## Monte Carlo susceptibility from measured validation results

`S11 - S11_center ≈ beta^T (x - x_center)` の局所線形モデルを Monte Carlo 摂動点の実測結果に fit します。`C_sigma = sigma_i^2 beta_i^2` を摂動分散で重み付けした susceptibility 寄与として出力します。


In [ ]:
def compute_mc_susceptibility(
    results_df,
    x_center,
    param_names,
    sigma_vec,
    objective_col="S11",
    center_row_index=0,
    ridge=1e-10,
    use_gaussian_weights=True,
):
    """Estimate sigma-weighted susceptibility from measured Monte Carlo samples."""
    x_center = np.asarray(x_center, dtype=float).reshape(-1)
    sigma_vec = np.asarray(sigma_vec, dtype=float).reshape(-1)
    X = results_df[param_names].to_numpy(dtype=float)
    y = results_df[objective_col].to_numpy(dtype=float)

    if X.shape[1] != x_center.size or sigma_vec.size != x_center.size:
        raise ValueError("x_center, sigma_vec, and param_names dimensions do not match.")
    if len(results_df) < 2:
        raise ValueError("At least the center row and one perturbation row are required.")

    y_center = float(y[center_row_index])
    dX = X - x_center[None, :]
    dy = y - y_center

    mask = np.ones(len(results_df), dtype=bool)
    mask[center_row_index] = False
    dX_fit = dX[mask]
    dy_fit = dy[mask]

    active = sigma_vec > 0
    if not np.any(active):
        raise ValueError("At least one sigma_vec component must be positive.")

    X_scaled = dX_fit[:, active] / sigma_vec[active]
    if use_gaussian_weights:
        radius2 = np.sum(X_scaled ** 2, axis=1)
        weights = np.exp(-0.5 * radius2)
    else:
        weights = np.ones(X_scaled.shape[0], dtype=float)

    sqrt_w = np.sqrt(weights)[:, None]
    A = X_scaled * sqrt_w
    b = dy_fit * sqrt_w[:, 0]

    lhs = A.T @ A + ridge * np.eye(A.shape[1])
    rhs = A.T @ b
    beta_scaled = np.linalg.solve(lhs, rhs)

    beta = np.zeros(x_center.size, dtype=float)
    beta[active] = beta_scaled / sigma_vec[active]

    S_raw = beta ** 2
    C_sigma = (sigma_vec ** 2) * S_raw
    sum_S = float(np.sum(S_raw))
    sum_C = float(np.sum(C_sigma))

    return {
        "beta": beta,
        "S_raw": S_raw,
        "S_raw_normalized": S_raw / sum_S if sum_S > 0 else np.zeros_like(S_raw),
        "C_sigma": C_sigma,
        "C_sigma_normalized": C_sigma / sum_C if sum_C > 0 else np.zeros_like(C_sigma),
        "sigma_vec": sigma_vec,
        "y_center": y_center,
        "n_fit_samples": int(mask.sum()),
        "weights": weights,
    }


susceptibility_result = compute_mc_susceptibility(
    results_df=df_output,
    x_center=x_center,
    param_names=PARAM_NAMES,
    sigma_vec=CUSTOM_PERTURBATION_STD_ARRAY,
    objective_col="S11",
    center_row_index=0,
    ridge=1e-10,
    use_gaussian_weights=True,
)

susceptibility_df = pd.DataFrame({
    "param": PARAM_NAMES,
    "sigma_x": susceptibility_result["sigma_vec"],
    "beta_dS11_dx": susceptibility_result["beta"],
    "S_raw": susceptibility_result["S_raw"],
    "S_raw_normalized": susceptibility_result["S_raw_normalized"],
    "C_sigma": susceptibility_result["C_sigma"],
    "C_sigma_normalized": susceptibility_result["C_sigma_normalized"],
})#.sort_values("C_sigma_normalized", ascending=False, ignore_index=True)

susceptibility_csv = run_dir / "mc_susceptibility.csv"
susceptibility_df.to_csv(susceptibility_csv, index=False)
print("Center S11:", susceptibility_result["y_center"])
print("Fit samples:", susceptibility_result["n_fit_samples"])
print(f"Saved Monte Carlo susceptibility CSV: {susceptibility_csv}")
display(susceptibility_df)


Center S11: -23.5975520234
Fit samples: 100
Saved Monte Carlo susceptibility CSV: T:\RAkizawa\HFSS_C2WR10\src\0618092513\mc_susceptibility.csv


,param,sigma_x,beta_dS11_dx,S_raw,S_raw_normalized,C_sigma,C_sigma_normalized
0,h1,0.10,0.363770,0.132328,0.000321,0.001323,0.001345
1,h2,0.10,-0.021671,0.000470,0.000001,0.000005,0.000005
2,h3,0.10,1.304894,1.702748,0.004129,0.017027,0.017304
3,h4,0.10,2.578394,6.648117,0.016123,0.066481,0.067559
4,h5,0.10,-2.899931,8.409601,0.020395,0.084096,0.085459
5,s1,0.01,4.296090,18.456390,0.044760,0.001846,0.001876
6,s2,0.02,-6.932111,48.054156,0.116540,0.019222,0.019533
7,s3,0.02,-10.736706,115.276866,0.279568,0.046111,0.046858
8,s4,0.02,-11.969692,143.273523,0.347465,0.057309,0.058239
9,s5,0.02,1.173996,1.378267,0.003343,0.000551,0.000560
